# 02 - 计划生成深度教程 (Plan Generation Deep Dive)

---
## 目录

1. [理论基础](#1-理论基础)
2. [Plan 数据结构](#2-plan-数据结构)
3. [前向规划](#3-前向规划)
4. [后向规划](#4-后向规划)
5. [层次规划](#5-层次规划)
6. [约束处理](#6-约束处理)
7. [计划验证](#7-计划验证)
8. [实战案例](#8-实战案例)
9. [高级技巧](#9-高级技巧)
10. [性能对比](#10-性能对比)

---

## 1. 理论基础

### 1.1 规划问题数学模型

规划可以形式化为搜索问题：

$$P = (S, A, T, s_0, G, C)$$

其中：
- $S$: 状态空间（所有可能的世界状态）
- $A$: 动作空间（可执行的操作）
- $T: S \times A \rightarrow S$: 状态转移函数
- $s_0 \in S$: 初始状态
- $G \subseteq S$: 目标状态集合
- $C: A \rightarrow \mathbb{R}^+$: 成本函数

### 1.2 规划目标

找到动作序列 $\pi = [a_1, ..., a_n]$ 使得：

$$T(...T(T(s_0, a_1), a_2)..., a_n) \in G$$

$$\text{minimize } \sum_{i=1}^{n} C(a_i)$$

### 1.3 规划策略对比

| 策略 | 搜索方向 | 优势 | 劣势 | 适用场景 |
|------|----------|------|------|----------|
| 前向 | $s_0 \rightarrow G$ | 直观、易实现 | 搜索空间大 | 清晰初始状态 |
| 后向 | $G \rightarrow s_0$ | 目标导向 | 需明确目标 | 清晰目标条件 |
| 层次 | 抽象→具体 | 分层抽象 | 设计复杂 | 复杂多阶段任务 |

### 1.4 时间复杂度分析

- **前向规划**: $O(b^d)$
  - $b$: 分支因子（每步可选动作数）
  - $d$: 解深度

- **后向规划**: $O(b^d)$
  - 通常分支因子较小（逆向推理）

- **层次规划**: $O(\sum_{i=1}^{l} b_i^{d_i})$
  - $l$: 抽象层级数
  - 每层独立规划，降低整体复杂度

In [ ]:
# =============================================================================
# 导入和初始化
# =============================================================================

import sys
sys.path.insert(0, '../src')

from task_decomposition import Task, TaskStatus, TaskPriority, TaskType
from plan_generation import (
    Plan, PlanStatus, PlanningStrategy,
    Constraint, ConstraintType,
    ForwardPlanner, BackwardPlanner, HierarchicalPlanner,
    PlanValidator, ValidationResult,
    create_planner, create_plan, format_plan
)

import json
from datetime import datetime, timedelta
from typing import List, Dict, Any
import time

print("✓ 所有模块导入成功")
print(f"✓ 当前时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---

## 2. Plan 数据结构

### 2.1 Plan 类完整结构

In [ ]:
# =============================================================================
# 创建基础计划
# =============================================================================

plan = Plan(
    goal="构建微服务架构的电商平台",
    metadata={
        "project": "E-commerce",
        "team": "Platform",
        "estimated_duration": "90 days",
        "budget": "$500,000"
    }
)

print("=== Plan 基础信息 ===")
print(f"\n计划ID: {plan.id}")
print(f"目标: {plan.goal}")
print(f"状态: {plan.status.value}")
print(f"创建时间: {plan.created_at}")
print(f"任务数: {len(plan.tasks)}")
print(f"约束数: {len(plan.constraints)}")

print(f"\n元数据:")
for key, value in plan.metadata.items():
    print(f"  {key}: {value}")

In [ ]:
# =============================================================================
# 添加任务到计划
# =============================================================================

# 创建多个任务
tasks = [
    Task(
        name="需求分析",
        description="收集和分析业务需求",
        priority=TaskPriority.HIGH,
        estimated_duration=5 * 8 * 3600  # 5个工作日
    ),
    Task(
        name="架构设计",
        description="设计系统架构和技术选型",
        priority=TaskPriority.HIGH,
        dependencies=[],
        estimated_duration=7 * 8 * 3600
    ),
    Task(
        name="服务开发",
        description="开发微服务模块",
        priority=TaskPriority.MEDIUM,
        estimated_duration=30 * 8 * 3600
    ),
    Task(
        name="测试上线",
        description="集成测试和部署上线",
        priority=TaskPriority.MEDIUM,
        estimated_duration=10 * 8 * 3600
    ),
]

# 建立依赖关系
tasks[2].add_dependency(tasks[1].id)  # 服务开发依赖架构设计
tasks[3].add_dependency(tasks[2].id)  # 测试依赖服务开发

# 添加到计划
for task in tasks:
    plan.add_task(task)

print("=== 计划任务列表 ===")
for i, task in enumerate(plan.tasks, 1):
    deps = f" (依赖: {len(task.dependencies)})" if task.dependencies else ""
    print(f"\n{i}. {task.name}{deps}")
    print(f"   描述: {task.description}")
    print(f"   优先级: {task.priority.value}")
    print(f"   预估时长: {task.estimated_duration / 3600 / 8:.1f} 工作日")

### 2.2 Plan 的计算属性

In [ ]:
# =============================================================================
# Plan 计算属性演示
# =============================================================================

# 模拟部分任务完成
plan.tasks[0].status = TaskStatus.COMPLETED  # 需求分析完成
plan.tasks[1].status = TaskStatus.IN_PROGRESS  # 架构设计中

print("=== Plan 计算属性 ===")

# 进度计算
print(f"\n1. 进度 (progress): {plan.progress:.1%}")
print(f"   计算方式: 已完成任务数 / 总任务数")
print(f"   已完成: {sum(1 for t in plan.tasks if t.status == TaskStatus.COMPLETED)}")
print(f"   总任务: {len(plan.tasks)}")

# 完成状态
print(f"\n2. 是否完成 (is_complete): {plan.is_complete}")
print(f"   条件: 所有任务状态为 COMPLETED")

# 失败检查
print(f"\n3. 是否有失败任务 (has_failed_tasks): {plan.has_failed_tasks}")
print(f"   条件: 存在任务状态为 FAILED")

# 可执行任务
ready = plan.get_ready_tasks()
print(f"\n4. 可执行任务 (get_ready_tasks): {len(ready)} 个")
for task in ready:
    print(f"   - {task.name}")
    print(f"     依赖已满足: {task.dependencies}")

# 执行顺序
print(f"\n5. 执行顺序 (get_execution_order):")
try:
    order = plan.get_execution_order()
    for i, task in enumerate(order, 1):
        status_icon = {
            TaskStatus.PENDING: "[ ]",
            TaskStatus.IN_PROGRESS: "[~]",
            TaskStatus.COMPLETED: "[x]"
        }.get(task.status, "[?]")
        print(f"   {i}. {status_icon} {task.name}")
except ValueError as e:
    print(f"   错误: {e}")

### 2.3 计划序列化

In [ ]:
# =============================================================================
# 序列化和反序列化
# =============================================================================

# 序列化
plan_dict = plan.to_dict()

print("=== Plan 序列化 ===")
print(f"\n序列化结果包含字段:")
for key in plan_dict.keys():
    print(f"  - {key}")

# 反序列化
restored_plan = Plan.from_dict(plan_dict)

print(f"\n=== 反序列化验证 ===")
print(f"原计划ID: {plan.id}")
print(f"还原计划ID: {restored_plan.id}")
print(f"ID匹配: {plan.id == restored_plan.id}")
print(f"任务数: {len(restored_plan.tasks)}")
print(f"目标: {restored_plan.goal}")

---

## 3. 前向规划 (Forward Planning)

### 3.1 算法原理

**前向规划** 从初始状态出发，逐步向目标状态推进。

```
算法 ForwardPlanning(goal):
    state = initial_state
    plan = []
    
    while not goal_reached(state):
        actions = get_applicable_actions(state)
        best_action = select_best_action(actions, goal)
        plan.append(best_action)
        state = apply(state, best_action)
    
    return plan
```

**特点**：
- ✓ 直观易懂
- ✓ 容易实现
- ✗ 搜索空间可能很大
- ✗ 可能产生无关动作

### 3.2 前向规划演示

In [ ]:
# =============================================================================
# 前向规划实战
# =============================================================================

planner = ForwardPlanner(max_tasks=10)

# 示例1: 简单目标
goal1 = "搭建个人博客网站"
plan1 = planner.generate(
    goal=goal1,
    context="使用Python和静态网站生成器"
)

print("=== 前向规划: 示例1 ===")
print(f"\n目标: {goal1}")
print(f"生成任务数: {len(plan1.tasks)}")
print(f"\n执行计划:")
for i, task in enumerate(plan1.tasks, 1):
    print(f"\n{i}. {task.name}")
    print(f"   {task.description}")
    print(f"   类型: {task.task_type.value} | 优先级: {task.priority.value}")

In [ ]:
# =============================================================================
# 示例2: 带约束的前向规划
# =============================================================================

# 创建约束
constraints = [
    Constraint(
        name="时间限制",
        constraint_type=ConstraintType.TIME,
        description="必须在2周内完成",
        parameters={"max_duration": 14 * 24 * 3600},
        is_hard=True
    ),
    Constraint(
        name="资源限制",
        constraint_type=ConstraintType.RESOURCE,
        description="只有1名开发人员",
        parameters={"max_developers": 1},
        is_hard=True
    ),
]

goal2 = "开发简单的任务管理APP"
plan2 = planner.generate(
    goal=goal2,
    context="个人项目，学习目的",
    constraints=constraints
)

print("\n=== 前向规划: 示例2 (带约束) ===")
print(f"\n目标: {goal2}")
print(f"约束数: {len(plan2.constraints)}")
for c in plan2.constraints:
    print(f"  - {c.name}: {c.description}")

print(f"\n生成的任务:")
for i, task in enumerate(plan2.tasks, 1):
    print(f"{i}. {task.name}")

---

## 4. 后向规划 (Backward Planning)

### 4.1 算法原理

**后向规划** 从目标状态出发，逆向推理到初始状态。

```
算法 BackwardPlanning(goal):
    goals = [goal]
    plan = []
    
    while not satisfied(goals, initial_state):
        current_goal = select_subgoal(goals)
        actions = get_achieving_actions(current_goal)
        best_action = select_best_action(actions)
        plan.prepend(best_action)  # 添加到开头
        goals.extend(get_preconditions(best_action))
        goals.remove(current_goal)
    
    return plan
```

**特点**：
- ✓ 目标导向
- ✓ 减少无关动作
- ✗ 需要明确目标状态
- ✗ 逆向推理可能不直观

### 4.2 后向规划演示

In [ ]:
# =============================================================================
# 后向规划实战
# =============================================================================

backward_planner = BackwardPlanner(max_tasks=10)

goal = "系统能够处理每秒10000次请求"

plan = backward_planner.generate(
    goal=goal,
    context="从零开始构建高并发系统"
)

print("=== 后向规划演示 ===")
print(f"\n目标状态: {goal}")
print(f"\n逆向推理的步骤:")
print("(从目标倒推: 要达到目标需要什么? → 为了那个需要什么? → ...)")

print(f"\n生成的执行顺序:")
for i, task in enumerate(plan.tasks, 1):
    print(f"{i}. {task.name}")
    print(f"   {task.description}")

---

## 5. 层次规划 (Hierarchical Planning)

### 5.1 算法原理

**层次规划** 在多个抽象层次上进行规划。

```
算法 HierarchicalPlanning(goal):
    # 抽象层
    high_level_plan = plan_at_abstract_level(goal)
    
    # 细化层
    for each task in high_level_plan:
        if not is_atomic(task):
            subtasks = refine_task(task)
            task.subtasks = subtasks
    
    return complete_plan
```

**HTN (Hierarchical Task Network)** 核心概念：
- **复合任务**: 可分解的任务
- **原始任务**: 原子操作，直接执行
- **方法**: 定义如何分解复合任务

### 5.2 层次规划演示

In [ ]:
# =============================================================================
# 层次规划实战
# =============================================================================

hier_planner = HierarchicalPlanner(max_phases=4, max_tasks_per_phase=5)

goal = "构建完整的DevOps流水线"

plan = hier_planner.generate(
    goal=goal,
    context="包括CI/CD、监控、日志管理等"
)

print("=== 层次规划演示 ===")
print(f"\n目标: {goal}")
print(f"生成任务数: {len(plan.tasks)}")

# 按阶段分组
phases = {}
for task in plan.tasks:
    phase = task.metadata.get("phase", "未分类")
    if phase not in phases:
        phases[phase] = []
    phases[phase].append(task)

print("\n=== 计划阶段划分 ===")
for phase_name, tasks in phases.items():
    print(f"\n【{phase_name}】")
    for i, task in enumerate(tasks, 1):
        print(f"  {i}. {task.name}")
        print(f"     {task.description}")

---

## 6. 约束处理

### 6.1 约束类型详解

In [ ]:
# =============================================================================
# 各种约束类型演示
# =============================================================================

plan = Plan(goal="开发企业级应用")

# 1. 时间约束
time_constraint = Constraint(
    name="项目截止日期",
    constraint_type=ConstraintType.TIME,
    description="必须在Q4结束前完成",
    parameters={
        "deadline": "2024-12-31",
        "max_duration": 90 * 24 * 3600  # 90天
    },
    is_hard=True
)

# 2. 资源约束
resource_constraint = Constraint(
    name="开发团队限制",
    constraint_type=ConstraintType.RESOURCE,
    description="最多5名开发人员",
    parameters={
        "max_developers": 5,
        "available_skills": ["Python", "React", "DevOps"]
    },
    is_hard=True
)

# 3. 依赖约束
dep_constraint = Constraint(
    name="技术依赖",
    constraint_type=ConstraintType.DEPENDENCY,
    description="某些任务需要特定库/服务先就绪",
    parameters={
        "required_services": ["database", "cache", "storage"]
    },
    is_hard=True
)

# 4. 顺序约束
order_constraint = Constraint(
    name="合规要求",
    constraint_type=ConstraintType.ORDERING,
    description="安全审查必须在部署前",
    parameters={
        "before_task": "deploy",
        "after_task": "security_review"
    },
    is_hard=True
)

# 5. 软约束 (偏好)
soft_constraint = Constraint(
    name="技术栈偏好",
    constraint_type=ConstraintType.RESOURCE,
    description="优先使用TypeScript而非JavaScript",
    parameters={"preferred_language": "TypeScript"},
    is_hard=False  # 软约束
)

constraints = [
    time_constraint, resource_constraint, dep_constraint,
    order_constraint, soft_constraint
]

for c in constraints:
    plan.add_constraint(c)

print("=== 计划约束演示 ===")
print(f"\n约束总数: {len(plan.constraints)}")

for c in plan.constraints:
    type_str = f"[{c.constraint_type.value}]"
    hard_str = "硬约束" if c.is_hard else "软约束"
    print(f"\n{type_str} {c.name} ({hard_str})")
    print(f"  描述: {c.description}")
    print(f"  参数: {json.dumps(c.parameters, ensure_ascii=False)}")

### 6.2 约束验证

In [ ]:
# =============================================================================
# 约束验证演示
# =============================================================================

validator = PlanValidator()

# 添加一些任务到计划
plan.add_task(Task(name="需求分析", estimated_duration=5*24*3600))
plan.add_task(Task(name="系统设计", estimated_duration=10*24*3600))
plan.add_task(Task(name="开发实现", estimated_duration=30*24*3600))

validation_result = validator.validate(plan)

print("=== 约束验证结果 ===")
print(f"\n计划有效: {validation_result.is_valid}")
print(f"验证分数: {validation_result.score:.2f} / 1.00")

if validation_result.issues:
    print("\n发现问题:")
    for issue in validation_result.issues:
        print(f"  ✗ {issue}")

if validation_result.warnings:
    print("\n警告:")
    for warning in validation_result.warnings:
        print(f"  ⚠ {warning}")

---

## 7. 计划验证

### 7.1 验证检查项

In [ ]:
# =============================================================================
# 各种验证场景
# =============================================================================

validator = PlanValidator()

test_cases = [
    {
        "name": "空计划",
        "plan": Plan(goal="测试"),
    },
    {
        "name": "有效计划",
        "plan": create_plan("有效目标", tasks=[
            Task(name="任务1"), Task(name="任务2")
        ]),
    },
    {
        "name": "循环依赖",
        "plan": None  # 稍后创建
    },
]

# 创建循环依赖计划
t1 = Task(name="A")
t2 = Task(name="B")
t1.dependencies = [t2.id]
t2.dependencies = [t1.id]
cyclic_plan = Plan(goal="循环依赖测试")
cyclic_plan.add_task(t1)
cyclic_plan.add_task(t2)
test_cases[2]["plan"] = cyclic_plan

print("=== 计划验证测试 ===")

for test in test_cases:
    result = validator.validate(test["plan"])
    
    status = "✓ 有效" if result.is_valid else "✗ 无效"
    print(f"\n{test['name']}: {status}")
    print(f"  分数: {result.score:.2f}")
    
    if result.issues:
        print("  问题:")
        for issue in result.issues:
            print(f"    - {issue}")
    
    if result.warnings:
        print("  警告:")
        for warning in result.warnings:
            print(f"    - {warning}")

---

## 8. 实战案例

### 8.1 完整项目规划

In [ ]:
# =============================================================================
# 实战: 企业数据中台项目规划
# =============================================================================

print("\n" + "="*60)
print("实战案例: 企业数据中台建设")
print("="*60)

goal = """建设企业级数据中台，实现数据资产的统一管理和价值挖掘。
核心能力包括: 数据采集、数据开发、数据治理、数据服务。
目标用户: 数据分析师、数据科学家、业务人员。
"""

# 使用层次规划
planner = HierarchicalPlanner(max_phases=5, max_tasks_per_phase=6)

# 添加约束
constraints = [
    Constraint(
        name="项目周期",
        constraint_type=ConstraintType.TIME,
        description="6个月完成一期",
        parameters={"max_duration": 180 * 24 * 3600},
        is_hard=True
    ),
    Constraint(
        name="团队规模",
        constraint_type=ConstraintType.RESOURCE,
        description="20人团队",
        parameters={"team_size": 20},
        is_hard=True
    ),
]

# 生成计划
plan = planner.generate(
    goal=goal.strip(),
    context="金融行业，日均数据量TB级",
    constraints=constraints
)

print("\n=== 生成计划概览 ===")
print(format_plan(plan, verbose=True))

# 验证计划
validator = PlanValidator()
validation = validator.validate(plan)

print("\n" + "="*60)
print("计划验证")
print("="*60)
print(f"有效性: {'✓ 通过' if validation.is_valid else '✗ 失败'}")
print(f"验证分数: {validation.score:.2f} / 1.00")

if validation.issues:
    print("\n需要修复的问题:")
    for issue in validation.issues:
        print(f"  - {issue}")

In [ ]:
# =============================================================================
# 计划统计分析
# =============================================================================

print("\n" + "="*60)
print("计划统计分析")
print("="*60)

# 按类型统计
type_counts = {}
for task in plan.tasks:
    ttype = task.task_type.value
    type_counts[ttype] = type_counts.get(ttype, 0) + 1

print("\n任务类型分布:")
for ttype, count in type_counts.items():
    print(f"  {ttype:15} {count:3} 个 ({count/len(plan.tasks)*100:.1f}%)")

# 按优先级统计
priority_counts = {}
for task in plan.tasks:
    priority = task.priority.value
    priority_counts[priority] = priority_counts.get(priority, 0) + 1

print("\n优先级分布:")
for priority, count in priority_counts.items():
    print(f"  {priority.upper():10} {count:3} 个")

# 工作量估算
total_estimated = sum(
    t.estimated_duration or 0 
    for t in plan.tasks
)

print(f"\n工作量估算:")
print(f"  总预估时长: {total_estimated / 3600 / 8:.1f} 人日")
print(f"  平均每人: {total_estimated / 3600 / 8 / 20:.1f} 人日")  # 假设20人团队

---

## 9. 高级技巧

### 9.1 计划合并

In [ ]:
# =============================================================================
# 高级技巧: 计划合并
# =============================================================================

print("\n=== 高级技巧: 计划合并 ===")

# 创建两个独立的计划
plan1 = create_planner("forward").generate("开发后端API")
plan2 = create_planner("forward").generate("开发前端界面")

print(f"\n计划1: {plan1.goal} ({len(plan1.tasks)} 任务)")
print(f"计划2: {plan2.goal} ({len(plan2.tasks)} 任务)")

# 合并计划
merged_plan = Plan(
    goal="全栈开发: 后端API + 前端界面",
    tasks=plan1.tasks + plan2.tasks,
    constraints=plan1.constraints + plan2.constraints
)

# 添加前端依赖后端的约束
if plan1.tasks and plan2.tasks:
    # 前端最后一个任务依赖后端完成
    # (这里简化处理，实际应该更智能)
    pass

print(f"\n合并后: {merged_plan.goal} ({len(merged_plan.tasks)} 任务)")
print(f"\n合并计划结构:")
print(format_plan(merged_plan))

### 9.2 计划优化

In [ ]:
# =============================================================================
# 计划优化演示
# =============================================================================

from plan_refinement import PlanOptimizer

# 创建一个有重复任务的计划
plan = Plan(goal="优化测试")
plan.add_task(Task(name="设计数据库"))
plan.add_task(Task(name="设计数据库"))  # 重复
plan.add_task(Task(name="开发API"))
plan.add_task(Task(name="开发API"))    # 重复
plan.add_task(Task(name="测试"))

print("\n=== 计划优化 ===")
print(f"\n优化前任务数: {len(plan.tasks)}")
for t in plan.tasks:
    print(f"  - {t.name}")

# 优化
optimizer = PlanOptimizer()
result = optimizer.optimize(plan)

print(f"\n优化后任务数: {len(plan.tasks)}")
for t in plan.tasks:
    print(f"  - {t.name}")

print(f"\n优化结果: {result.success}")
print(f"变更: {result.changes_made}")

---

## 10. 性能对比

### 10.1 规划策略性能对比

In [ ]:
# =============================================================================
# 性能对比测试
# =============================================================================

test_goals = [
    "构建个人博客",
    "开发电商平台",
    "搭建微服务架构",
    "建设数据中台",
]

strategies = [
    ("Forward", ForwardPlanner()),
    ("Backward", BackwardPlanner()),
    ("Hierarchical", HierarchicalPlanner()),
]

print("\n" + "="*60)
print("规划策略性能对比")
print("="*60)

results = {}

for goal in test_goals:
    print(f"\n目标: {goal}")
    results[goal] = {}
    
    for strategy_name, planner in strategies:
        start = time.time()
        plan = planner.generate(goal)
        elapsed = time.time() - start
        
        results[goal][strategy_name] = {
            "time": elapsed,
            "tasks": len(plan.tasks)
        }
        
        print(f"  {strategy_name:12} {elapsed*1000:6.2f}ms  {len(plan.tasks)} 任务")

# 汇总统计
print("\n" + "="*60)
print("平均性能")
print("="*60)

for strategy_name, _ in strategies:
    times = [results[g][strategy_name]["time"] for g in test_goals]
    tasks = [results[g][strategy_name]["tasks"] for g in test_goals]
    
    avg_time = sum(times) / len(times) * 1000
    avg_tasks = sum(tasks) / len(tasks)
    
    print(f"\n{strategy_name}:")
    print(f"  平均生成时间: {avg_time:.2f}ms")
    print(f"  平均任务数: {avg_tasks:.1f}")

---

## 11. 最佳实践

### 11.1 规划技巧

In [ ]:
# =============================================================================
# 最佳实践总结
# =============================================================================

best_practices = {
    "目标定义": [
        "使用SMART原则 (具体、可衡量、可达成、相关、有时限)",
        "明确成功标准和验收条件",
        "提供足够的上下文信息",
    ],
    "策略选择": [
        "前向规划: 初始状态清晰时",
        "后向规划: 目标状态明确时",
        "层次规划: 复杂多阶段项目",
    ],
    "约束管理": [
        "区分硬约束和软约束",
        "避免过度约束",
        "定期验证约束合理性",
    ],
    "计划验证": [
        "检查循环依赖",
        "验证所有依赖有效",
        "确保计划可达",
    ],
    "迭代优化": [
        "初始计划不求完美",
        "执行中持续优化",
        "保持一定的灵活性",
    ],
}

print("\n" + "="*60)
print("计划生成最佳实践")
print("="*60)

for category, practices in best_practices.items():
    print(f"\n{category}:")
    for i, practice in enumerate(practices, 1):
        print(f"  {i}. {practice}")

---

## 12. 练习题

### 练习: 为你的项目制定计划

尝试为下面的目标制定一个完整的计划：

In [ ]:
# =============================================================================
# 练习区域
# =============================================================================

# TODO: 替换为你的目标
my_goal = "[你的项目目标]"

# TODO: 选择规划策略
# planner = create_planner("hierarchical")  # forward, backward, hierarchical

# TODO: 生成计划
# my_plan = planner.generate(my_goal)

# TODO: 验证计划
# validator = PlanValidator()
# result = validator.validate(my_plan)

# TODO: 查看结果
# print(format_plan(my_plan, verbose=True))

print("\nTODO: 完成上面的练习!")

---

## 13. 总结

### 本教程覆盖的内容

1. ✓ Plan 数据结构的完整使用
2. ✓ 前向、后向、层次三种规划策略
3. ✓ 约束处理和验证
4. ✓ 计划验证技术
5. ✓ 实战案例: 数据中台项目
6. ✓ 性能对比分析
7. ✓ 最佳实践

### 下一步学习

- 📖 **03_PlanExecution_tutorial**: 学习如何执行计划
- 📖 **知识点.md**: 深入理解算法原理

### 关键要点

- **前向规划**: 从现状推导到目标，适合初始状态明确的场景
- **后向规划**: 从目标倒推到现状，适合目标清晰的需求
- **层次规划**: 多层抽象细化，适合复杂项目管理
- **约束验证**: 确保计划可行性和质量

---

**恭喜完成本教程! 🎉**